# 🔙 Backtracking -- Runnable Notebook

Companion to [`README.md`](README.md) and the interactive
[`21_backtracking_lesson.html`](21_backtracking_lesson.html).

Every backtracking problem in **Blind 75** and **NeetCode 150**, built from one template
(**choose → explore → un-choose**), and wherever the choice is binary -- "is this fixed element in the
answer or not" -- written as **include/exclude** rather than a start-index loop. Run each cell top to bottom -- every function prints its result and
`assert`s the expected answer, and two cells deliberately show the classic bugs so you can see *why* the
un-choose step and the `path[:]` copy matter.

## 1. The template -- Subsets ([LC 78](https://leetcode.com/problems/subsets/))

**Problem in plain language:** you're given a list of distinct numbers. Return **every possible subset**
of it (the "power set") -- including the empty subset and the full list itself. The order of numbers
inside a subset doesn't matter, and you shouldn't return the same subset twice.
Example: `[1, 2]` -> `[[], [1], [2], [1, 2]]`.

**Include / exclude:** a binary decision at every index -- either `nums[i]` is in the answer or it isn't.
`backtrack(i)` tries both: append `nums[i]` and recurse into `i + 1` (include), pop it, then recurse into
`i + 1` again without it (exclude). The recursion tree has exactly `2^n` leaves, one per subset, and that's
the only place anything gets recorded.

In [1]:
def subsets(nums):                                    # LC 78
    out, path = [], []
    def backtrack(i):
        if i == len(nums):                     # base case: decided every element
            out.append(path[:])                # record a COPY (see the bug demo below)
            return
        path.append(nums[i])                   # INCLUDE nums[i]
        backtrack(i + 1)
        path.pop()                             # UN-CHOOSE

        backtrack(i + 1)                       # EXCLUDE nums[i]
    backtrack(0)
    return out


a = subsets([1, 2, 3])
print("subsets([1,2,3]):", a)
assert len(a) == 8                                             # 2^3 subsets
assert [] in a and [1, 2, 3] in a

subsets([1,2,3]): [[1, 2, 3], [1, 2], [1, 3], [1], [2, 3], [2], [3], []]


### Step-by-step trace: why the leaves land where they do

Walking `nums = [1, 2, 3]`. Each call to `backtrack(i)` does two things at index `i`: first explore
**with** `nums[i]` in the path, then pop it and explore **without** it -- both branches call
`backtrack(i + 1)`. Nothing is recorded until `i == 3` (a leaf) -- there are exactly `2^3 = 8` of them.

| Step | Action | `path` after | `out` after |
|---|---|---|---|
| 1 | `backtrack(0)` starts | `[]` | `[]` |
| 2 | INCLUDE `1` | `[1]` | `[]` |
| 3 | `backtrack(1)` -> INCLUDE `2` | `[1,2]` | `[]` |
| 4 | `backtrack(2)` -> INCLUDE `3` | `[1,2,3]` | `[]` |
| 5 | `backtrack(3)`: `i==3` -> **RECORD** | `[1,2,3]` | `[[1,2,3]]` |
| 6 | UN-CHOOSE (pop `3`) | `[1,2]` | `[[1,2,3]]` |
| 7 | `backtrack(3)` again (EXCLUDE `3`): `i==3` -> **RECORD** | `[1,2]` | `[..., [1,2]]` |
| 8 | back in `backtrack(1)`: UN-CHOOSE (pop `2`) | `[1]` | `[..., [1,2]]` |
| 9 | `backtrack(2)` again (EXCLUDE `2`) -> INCLUDE `3` | `[1,3]` | `[..., [1,2]]` |
| 10 | `backtrack(3)`: **RECORD** | `[1,3]` | `[..., [1,3]]` |
| 11 | UN-CHOOSE (pop `3`) | `[1]` | `[..., [1,3]]` |
| 12 | `backtrack(3)` again (EXCLUDE `3`): **RECORD** | `[1]` | `[..., [1]]` |
| 13 | back in `backtrack(0)`: UN-CHOOSE (pop `1`) | `[]` | `[..., [1]]` |
| 14 | `backtrack(1)` again (EXCLUDE `1`) -> INCLUDE `2` | `[2]` | `[..., [1]]` |
| 15 | `backtrack(2)` -> INCLUDE `3` | `[2,3]` | `[..., [1]]` |
| 16 | `backtrack(3)`: **RECORD** | `[2,3]` | `[..., [2,3]]` |
| 17 | UN-CHOOSE (pop `3`) | `[2]` | `[..., [2,3]]` |
| 18 | `backtrack(3)` again: **RECORD** | `[2]` | `[..., [2]]` |
| 19 | UN-CHOOSE (pop `2`) | `[]` | `[..., [2]]` |
| 20 | `backtrack(2)` again (EXCLUDE `2`) -> INCLUDE `3` | `[3]` | `[..., [2]]` |
| 21 | `backtrack(3)`: **RECORD** | `[3]` | `[..., [3]]` |
| 22 | UN-CHOOSE (pop `3`) | `[]` | `[..., [3]]` |
| 23 | `backtrack(3)` again (EXCLUDE `3`): **RECORD** | `[]` | `[..., []]` |

Final `out`: `[[1,2,3], [1,2], [1,3], [1], [2,3], [2], [3], []]` -- 8 leaves, 8 subsets, every one of them
recorded at a fixed depth (`i == 3`).

### The two classic bugs, on purpose

1. **Missing un-choose** -- without `path.pop()` after the INCLUDE branch, the EXCLUDE branch that follows
   still has `nums[i]` sitting in `path`, so every "excluded" subset wrongly contains it too.
2. **Recording a reference** -- `out.append(path)` stores the *same* list eight times; by the end it's empty.

In [2]:
def subsets_missing_pop(nums):
    out, path = [], []
    def backtrack(i):
        if i == len(nums):
            out.append(path[:]); return
        path.append(nums[i])
        backtrack(i + 1)
        # path.pop()   <-- forgotten: EXCLUDE branch below still sees nums[i] in path
        backtrack(i + 1)
    backtrack(0)
    return out

def subsets_no_copy(nums):
    out, path = [], []
    def backtrack(i):
        if i == len(nums):
            out.append(path)                    # reference, not a copy
            return
        path.append(nums[i]); backtrack(i + 1); path.pop()
        backtrack(i + 1)
    backtrack(0)
    return out

bug1 = subsets_missing_pop([1, 2, 3])
bug2 = subsets_no_copy([1, 2, 3])
print("missing pop  :", bug1)
print("no copy      :", bug2)
assert len(bug1) == 8 and [1, 3] not in bug1      # right count, wrong contents: [1,3] never appears
assert bug2 == [[]] * 8                            # eight references to one list, which is empty at the end

missing pop  : [[1, 2, 3], [1, 2, 3], [1, 2, 3, 3], [1, 2, 3, 3], [1, 2, 3, 3, 2, 3], [1, 2, 3, 3, 2, 3], [1, 2, 3, 3, 2, 3, 3], [1, 2, 3, 3, 2, 3, 3]]
no copy      : [[], [], [], [], [], [], [], []]


## 2. Subsets II -- duplicates in the input ([LC 90](https://leetcode.com/problems/subsets-ii/))

**Problem in plain language:** same task as Subsets ([LC 78](https://leetcode.com/problems/subsets/)) -- return every possible subset -- but now
the input list can contain **duplicate numbers** (e.g. `[1, 2, 2]`). Return every **distinct** subset
only; don't output the same subset twice just because a number was repeated in the input.
Example: `[1, 2, 2]` -> `[[], [1], [1,2], [1,2,2], [2], [2,2]]` (not 8 subsets, only 6 distinct ones).

Same include/exclude binary decision as plain Subsets -- but the EXCLUDE branch can no longer just move to
`i + 1`. Excluding `nums[i]` and excluding its next duplicate are *the same decision*, so EXCLUDE has to
skip past every remaining copy of that value in one go, or the two copies would each spawn their own
"excluded" branch and produce the same subsets twice. Sort first so duplicates sit next to each other.

In [3]:
def subsets_with_dup(nums):                          # LC 90
    nums = sorted(nums)                                  # duplicates become adjacent
    n = len(nums)
    out, path = [], []
    def backtrack(i):
        if i == n:
            out.append(path[:]); return
        path.append(nums[i])                             # INCLUDE nums[i]
        backtrack(i + 1)
        path.pop()

        j = i + 1
        while j < n and nums[j] == nums[i]:               # EXCLUDE nums[i] -- and every
            j += 1                                        # duplicate of it, same value
        backtrack(j)
    backtrack(0)
    return out


res = subsets_with_dup([1, 2, 2])
print("subsets of [1,2,2]:", res)
assert sorted(map(sorted, res)) == sorted(map(sorted, [[], [1], [1, 2], [1, 2, 2], [2], [2, 2]]))
assert len(res) == len({tuple(s) for s in res})   # no duplicates

# Without the skip you get duplicate subsets -- plain subsets() treats the two 2s as different elements.
plain = subsets([1, 2, 2])
print("without the skip:", len(plain), "subsets,", len({tuple(s) for s in plain}), "distinct")
assert len(plain) == 8 and len({tuple(s) for s in plain}) == 6

subsets of [1,2,2]: [[1, 2, 2], [1, 2], [1], [2, 2], [2], []]
without the skip: 8 subsets, 6 distinct


## 3. Combinations & Combination Sum ([LC 77](https://leetcode.com/problems/combinations/), [LC 39](https://leetcode.com/problems/combination-sum/), [LC 40](https://leetcode.com/problems/combination-sum-ii/))

**Problem in plain language -- three separate problems here:**
- **Combinations ([LC 77](https://leetcode.com/problems/combinations/)):** given numbers `n` and `k`, return every way to **choose `k` numbers out of
  `1..n`**, where order doesn't matter (`{1,2}` and `{2,1}` count as the same choice, so only one is kept).
- **Combination Sum ([LC 39](https://leetcode.com/problems/combination-sum/)):** given a list of candidate numbers and a `target`, return every combination
  of candidates that **adds up exactly to `target`**. You're allowed to reuse the same candidate as many
  times as you like.
- **Combination Sum II ([LC 40](https://leetcode.com/problems/combination-sum-ii/)):** same as Combination Sum, but each number in the input list can be used
  **at most once** (even if it appears more than once in the input), and duplicate combinations in the
  output aren't allowed.

Same include/exclude binary decision as Subsets, with a different base case: record when the path reaches
length `k` or sum `target`, instead of at a fixed depth.

- **`backtrack(i, ...)` on INCLUDE** (stay at index `i`) lets a candidate be **reused**; **`backtrack(i + 1, ...)`**
  uses each element **once**.
- Before choosing to INCLUDE a candidate, check it actually fits (`remaining - candidates[i] >= 0`) --
  that's an `O(1)` guard that never even makes the doomed recursive call, the include/exclude equivalent of
  the sorted-loop's `break`.
- Combination Sum needs **no sorting at all** -- the fit-check works on candidates in any order. Combination
  Sum II still sorts, but only so the duplicate-skip in the EXCLUDE branch (same trick as Subsets II) can
  find adjacent duplicates.

In [4]:
def combine(n, k):                                    # LC 77
    out, path = [], []
    def backtrack(num):
        if len(path) == k:
            out.append(path[:]); return
        if num > n:
            return
        path.append(num)                                  # INCLUDE num
        backtrack(num + 1)
        path.pop()

        backtrack(num + 1)                                 # EXCLUDE num
    backtrack(1)
    return out


def combination_sum(candidates, target):                  # LC 39 -- unlimited reuse
    out, path = [], []
    def backtrack(i, remaining):
        if remaining == 0:
            out.append(path[:]); return
        if i == len(candidates):
            return
        if remaining - candidates[i] >= 0:                 # only recurse if it doesn't overshoot
            path.append(candidates[i])
            backtrack(i, remaining - candidates[i])        # `i`, not `i + 1`: INCLUDE may reuse
            path.pop()
        backtrack(i + 1, remaining)                        # EXCLUDE candidates[i] entirely
    backtrack(0, target)
    return out


def combination_sum2(candidates, target):                 # LC 40 -- each element once, no duplicate combos
    candidates = sorted(candidates)
    n = len(candidates)
    out, path = [], []
    def backtrack(i, remaining):
        if remaining == 0:
            out.append(path[:]); return
        if i == n:
            return
        if remaining - candidates[i] >= 0:
            path.append(candidates[i])
            backtrack(i + 1, remaining - candidates[i])   # `i + 1`: INCLUDE at most once
            path.pop()
        j = i + 1
        while j < n and candidates[j] == candidates[i]:    # EXCLUDE candidates[i] -- and every
            j += 1                                         # duplicate of it (same trick as Subsets II)
        backtrack(j, remaining)
    backtrack(0, target)
    return out


c = combine(4, 2)
print("combine(4, 2):", c)
assert sorted(map(sorted, c)) == sorted(map(sorted, [[1,2],[1,3],[1,4],[2,3],[2,4],[3,4]]))

cs = combination_sum([2, 3, 6, 7], 7)
print("combination_sum([2,3,6,7], 7):", cs)
assert sorted(map(sorted, cs)) == sorted(map(sorted, [[2, 2, 3], [7]]))

cs2 = combination_sum2([10, 1, 2, 7, 6, 1, 5], 8)
print("combination_sum2([10,1,2,7,6,1,5], 8):", cs2)
assert sorted(map(sorted, cs2)) == sorted(map(sorted, [[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]]))

combine(4, 2): [[1, 2], [1, 3], [1, 4], [2, 3], [2, 4], [3, 4]]
combination_sum([2,3,6,7], 7): [[2, 2, 3], [7]]
combination_sum2([10,1,2,7,6,1,5], 8): [[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]]


### Step-by-step: what this cell actually does

All three functions share one shape -- `backtrack(i, remaining)` (or, for `combine`, `backtrack(num)`)
makes a binary choice at index `i`: INCLUDE it (subtract it from `remaining`, and either stay at `i` or
move to `i + 1` depending on whether reuse is allowed) or EXCLUDE it (move on, skipping duplicates for
Combination Sum II). They differ in exactly the same two places as the Subsets pair above: whether the
INCLUDE branch's recursive call reuses index `i`, and whether there's a duplicate-skip in the EXCLUDE branch.

#### `combination_sum([2, 3, 6, 7], 7)` -- reuse allowed

Sorted candidates aren't required here; `[2, 3, 6, 7]` is already sorted for readability. `backtrack(i, remaining)`:
if `remaining == 0`, record; if `i == len(candidates)`, dead end; otherwise, only take the INCLUDE branch if
`candidates[i]` actually fits, recursing with **`backtrack(i, ...)`** (same index -- may reuse), then always
also try the EXCLUDE branch, `backtrack(i + 1, remaining)`.

| Step | Action | `path` after | `remaining` after |
|---|---|---|---|
| 1 | `backtrack(0, 7)` starts | `[]` | 7 |
| 2 | INCLUDE `2` (fits: `7-2=5`) | `[2]` | 5 |
| 3 | `backtrack(0, 5)` -> INCLUDE `2` again (same index: reuse) | `[2, 2]` | 3 |
| 4 | `backtrack(0, 3)` -> INCLUDE `2` again | `[2, 2, 2]` | 1 |
| 5 | `backtrack(0, 1)`: `2 > 1` -- INCLUDE branch never recurses; EXCLUDE -> `backtrack(1, 1)`: `3 > 1`, `6 > 1`, `7 > 1` -- every INCLUDE skipped, `i` reaches `4 == len(candidates)` -- dead end | `[2, 2, 2]` | 1 |
| 6 | unwind: pop back to `path=[2,2]`, `remaining=3`; EXCLUDE the third `2` -> `backtrack(1, 3)` | `[2, 2]` | 3 |
| 7 | INCLUDE `3` (fits: `3-3=0`) | `[2, 2, 3]` | 0 |
| 8 | `backtrack(1, 0)`: `remaining == 0` -> **RECORD `[2, 2, 3]`** | `[2, 2, 3]` | 0 |
| 9 | ...unwind, EXCLUDE branches at every level try `3`, `6`, `7` on their own (`[3]`, `[3,3]`, `[6]`, ...) -- all overshoot or fall short except one | | |
| 10 | eventually `backtrack(3, 7)`: INCLUDE `7` (fits: `7-7=0`) -> `backtrack(3, 0)` -> **RECORD `[7]`** | `[7]` | 0 |

Final `cs = [[2, 2, 3], [7]]` -- matches the assert. The fit-check before choosing INCLUDE is what keeps
the tree from ever descending into a branch that's already overshot; it plays the same role the sorted
`break` played in the start-index version, but per-candidate instead of cutting off the whole remaining loop
at once (measured in section 9: this version does more calls than the old `break` version, because EXCLUDE
still has to individually rule out each larger candidate rather than bailing on all of them at once).

#### `combination_sum2([10, 1, 2, 7, 6, 1, 5], 8)` -- each slot once, duplicates skipped

Sorted candidates: `[1, 1, 2, 5, 6, 7, 10]` (indices `0..6`), target `8`.

| Step | Action | `path` after | `remaining` after |
|---|---|---|---|
| 1 | `backtrack(0, 8)` starts | `[]` | 8 |
| 2 | INCLUDE `candidates[0]=1` (fits) | `[1]` | 7 |
| 3 | `backtrack(1, 7)` -> INCLUDE `candidates[1]=1` (fits; `i==1` is not `> start` in the old sense -- it's just "the next index", so this is a fresh choice, not a skipped duplicate) | `[1, 1]` | 6 |
| 4 | `backtrack(2, 6)` -> INCLUDE `candidates[2]=2` | `[1, 1, 2]` | 4 |
| 5 | ...INCLUDE `5` overshoots (`4-5<0`, never recurses); EXCLUDE `5` -> INCLUDE `6` overshoots too; EXCLUDE `6` -> INCLUDE `7` overshoots; EXCLUDE `7` -> INCLUDE `10` overshoots; EXCLUDE `10` -> `i==7==n` dead end | `[1, 1, 2]` | 4 |
| 6 | unwind to `path=[1,1]`, `remaining=6`; EXCLUDE `candidates[2]=2` -> `backtrack(3, 6)` | `[1, 1]` | 6 |
| 7 | ...similar explore/prune through `5`, `6` -> eventually INCLUDE `candidates[4]=6` (fits: `6-6=0`) | `[1, 1, 6]` | 0 |
| 8 | `remaining == 0` -> **RECORD `[1, 1, 6]`** | `[1, 1, 6]` | 0 |
| 9 | unwind further; EXCLUDE `candidates[1]=1` -- `candidates[2]=2 != candidates[1]=1`, so `j` only advances past `candidates[1]` itself, landing on index `2` (no duplicates to skip here since only `candidates[0]` and `candidates[1]` are equal, and both were already explored as INCLUDE choices, not skipped) | | |
| 10 | back at the top level, EXCLUDE `candidates[0]=1` skips **both** physical `1`s (`j` advances from `1` to `2`) -- that's the dedup: the whole `[1, ...]` subtree was already fully produced starting from index `0`, so index `1` never gets to start a fresh branch as the *first* pick | `[]` | 8 |
| 11 | continuing from index `2`: INCLUDE `candidates[2]=2` (`8-2=6`) -> INCLUDE `candidates[3]=5` (fits: `6-5=1`) -> `[2, 5]`, but every next candidate overshoots the remaining `1` -- dead end; unwind to `path=[2]`, `remaining=6`; EXCLUDE `candidates[3]=5` -> INCLUDE `candidates[4]=6` (fits: `6-6=0`) -> **RECORD `[2, 6]`** | `[2, 6]` | 0 |

Final `cs2 = [[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]]` -- matches the assert (run the cell above to see the
full printed list; the table above shows the mechanism, not every one of the ~190 recursive calls).

#### Mental model

- Both are the same choose -> explore -> un-choose recursion as every other problem in this notebook; only
  the recursive call's index (`i` vs `i + 1`) on INCLUDE, and the duplicate-skip in EXCLUDE, change the
  semantics from "reuse allowed" to "each slot once, no duplicate combos."
- `remaining` is the shrinking target, not a count -- recursion depth is bounded by
  `target / smallest candidate`.
- The fit-check (`remaining - candidates[i] >= 0`) before INCLUDE is what stops the recursion from ever
  entering an already-doomed branch -- section 9 measures exactly how much that saves.

> **Why the rest of this notebook stays start-index / `used`-array / direct loop:** include/exclude only
> works when the choice at each step is *binary* -- "is this one fixed array element in the answer or not".
> Once the choice becomes "which unused element goes next" (Permutations), "which of 4 directions" (Word
> Search), "how long is the next piece" (Palindrome Partitioning), "which letter for this digit" (Letter
> Combinations), or "which column" (N-Queens), there's no single element to include or exclude -- so these
> stay as start-index / `used`-array / direct-loop backtracking.

## 4. Permutations ([LC 46](https://leetcode.com/problems/permutations/), [LC 47](https://leetcode.com/problems/permutations-ii/))

**Problem in plain language:**
- **Permutations ([LC 46](https://leetcode.com/problems/permutations/)):** given a list of **distinct** numbers, return every possible **ordering**
  (arrangement) of them. Unlike subsets/combinations, order matters here -- `[1,2,3]` and `[3,2,1]` are
  different, separate answers.
- **Permutations II ([LC 47](https://leetcode.com/problems/permutations-ii/)):** same task, but the input can contain **duplicate** numbers. Return every
  **distinct** arrangement -- don't output the same ordering twice.

Order matters, so the start index is wrong here -- every level may pick **any** unused element. A `used`
array is the second piece of state, and it must be un-chosen too.

In [5]:
def permute(nums):                                   # LC 46
    out, path, used = [], [], [False] * len(nums)
    def backtrack():
        if len(path) == len(nums):
            out.append(path[:]); return
        for i in range(len(nums)):
            if used[i]:
                continue                                 # already on the path
            used[i] = True;  path.append(nums[i])        # CHOOSE (two pieces of state)
            backtrack()                                  # EXPLORE
            path.pop();      used[i] = False             # UN-CHOOSE (both pieces)
    backtrack()
    return out


def permute_unique(nums):                            # LC 47 -- input may contain duplicates
    nums = sorted(nums)
    out, path, used = [], [], [False] * len(nums)
    def backtrack():
        if len(path) == len(nums):
            out.append(path[:]); return
        for i in range(len(nums)):
            if used[i]:
                continue
            if i > 0 and nums[i] == nums[i - 1] and not used[i - 1]:
                continue                                 # twin to the left was already tried at this level
            used[i] = True;  path.append(nums[i])
            backtrack()
            path.pop();      used[i] = False
    backtrack()
    return out


p = permute([1, 2, 3])
print("permute([1,2,3]):", p)
assert len(p) == 6 and len({tuple(x) for x in p}) == 6     # 3! distinct orderings
assert [2, 1, 3] in p and [3, 2, 1] in p

pu = permute_unique([1, 1, 2])
print("permute_unique([1,1,2]):", pu)
assert pu == [[1, 1, 2], [1, 2, 1], [2, 1, 1]]

permute([1,2,3]): [[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]
permute_unique([1,1,2]): [[1, 1, 2], [1, 2, 1], [2, 1, 1]]


## 5. Word Search -- backtracking on a grid ([LC 79](https://leetcode.com/problems/word-search/))

**Problem in plain language:** you're given a 2D grid of letters and a target word. Starting from any
cell, can you spell out the word by moving one step at a time to a horizontally or vertically adjacent
cell, **without reusing the same cell twice** in one path? Return `True` if it's possible, `False`
otherwise.

The cell itself is the state: overwrite it with `"#"` on the way in, restore it on the way out. This variant
answers *yes / no*, so it returns `True` up the chain and `or` short-circuits at the first success.

In [6]:
def exist(board, word):                              # LC 79
    R, C = len(board), len(board[0])
    def dfs(r, c, k):                                    # k = index into `word` we must match at (r, c)
        if k == len(word):
            return True
        if not (0 <= r < R and 0 <= c < C) or board[r][c] != word[k]:
            return False
        saved, board[r][c] = board[r][c], "#"            # CHOOSE: mark the cell as on-path
        found = (dfs(r + 1, c, k + 1) or dfs(r - 1, c, k + 1) or
                 dfs(r, c + 1, k + 1) or dfs(r, c - 1, k + 1))
        board[r][c] = saved                              # UN-CHOOSE: always restore
        return found
    return any(dfs(r, c, 0) for r in range(R) for c in range(C))


board = [["A", "B", "C", "E"],
         ["S", "F", "C", "S"],
         ["A", "D", "E", "E"]]
snapshot = [row[:] for row in board]

for w, expected in [("ABCCED", True), ("SEE", True), ("ABCB", False)]:
    got = exist(board, w)
    print(f"exist({w!r}) = {got}")
    assert got == expected
assert board == snapshot          # every cell was restored, even along failed paths

exist('ABCCED') = True
exist('SEE') = True
exist('ABCB') = False


## 6. Palindrome Partitioning -- choose where to cut ([LC 131](https://leetcode.com/problems/palindrome-partitioning/))

**Problem in plain language:** given a string, cut it into pieces such that **every piece reads the same
forwards and backwards** (a palindrome). Return every possible way to make such a split.
Example: `"aab"` → `[["a","a","b"], ["aa","b"]]`.

The choice at each step is *how long the next piece is*. Only palindromic pieces are allowed, so the validity
check sits before the choose line and prunes everything else.

In [7]:
def partition(s):                                    # LC 131
    out, path = [], []
    def backtrack(start):
        if start == len(s):
            out.append(path[:]); return
        for end in range(start + 1, len(s) + 1):
            piece = s[start:end]
            if piece == piece[::-1]:                     # PRUNE: skip non-palindromic pieces
                path.append(piece); backtrack(end); path.pop()
    backtrack(0)
    return out


pp = partition("aab")
print("partition('aab'):", pp)
assert pp == [["a", "a", "b"], ["aa", "b"]]
assert all("".join(p) == "aab" for p in pp)      # every partition reassembles the input

partition('aab'): [['a', 'a', 'b'], ['aa', 'b']]


## 7. Letter Combinations of a Phone Number -- a Cartesian product ([LC 17](https://leetcode.com/problems/letter-combinations-of-a-phone-number/))

**Problem in plain language:** you're given a string of digits `2`-`9`, like an old phone keypad, where
each digit maps to a few letters (`2 -> "abc"`, `3 -> "def"`, ...). Return every possible letter
combination you could type by picking **one letter per digit**, keeping the digits' order.
Example: `"23"` → `["ad","ae","af","bd","be","bf","cd","ce","cf"]`.

No start index and no `used` set: every position is independent, and the tree's branching factor at depth `d`
is simply how many letters digit `d` maps to.

In [8]:
def letter_combinations(digits):                     # LC 17
    if not digits:
        return []
    phone = {"2": "abc", "3": "def", "4": "ghi", "5": "jkl",
             "6": "mno", "7": "pqrs", "8": "tuv", "9": "wxyz"}
    out, path = [], []
    def backtrack(i):
        if i == len(digits):
            out.append("".join(path)); return
        for ch in phone[digits[i]]:
            path.append(ch); backtrack(i + 1); path.pop()
    backtrack(0)
    return out


lc = letter_combinations("23")
print("letter_combinations('23'):", lc)
assert lc == ["ad", "ae", "af", "bd", "be", "bf", "cd", "ce", "cf"]
assert letter_combinations("") == []
assert len(letter_combinations("79")) == 4 * 4        # branching factors multiply

letter_combinations('23'): ['ad', 'ae', 'af', 'bd', 'be', 'bf', 'cd', 'ce', 'cf']


## 8. N-Queens -- constraint satisfaction with O(1) pruning sets ([LC 51](https://leetcode.com/problems/n-queens/))

**Problem in plain language:** place `n` chess queens on an `n x n` board so that **no two queens attack
each other** (no two share a row, a column, or a diagonal). Return every valid way to place them (each
answer shown as a board with `Q` for a queen and `.` for empty).

One queen per row; the choice is the column. Three sets make the safety check `O(1)`: every `\` diagonal shares
one `r - c`, every `/` diagonal shares one `r + c`. Four pieces of state to choose, four to un-choose.

In [9]:
def solve_n_queens(n):                               # LC 51
    out, cols, diag, anti = [], set(), set(), set()
    board = [["."] * n for _ in range(n)]
    def backtrack(r):
        if r == n:
            out.append(["".join(row) for row in board]); return
        for c in range(n):
            if c in cols or (r - c) in diag or (r + c) in anti:
                continue                                 # PRUNE: attacked by a queen above
            cols.add(c); diag.add(r - c); anti.add(r + c); board[r][c] = "Q"      # CHOOSE
            backtrack(r + 1)                                                      # EXPLORE
            cols.remove(c); diag.remove(r - c); anti.remove(r + c); board[r][c] = "."   # UN-CHOOSE
    backtrack(0)
    return out


sols = solve_n_queens(4)
print("4-queens solutions:")
for s in sols:
    print("   ", s)
assert sols == [[".Q..", "...Q", "Q...", "..Q."], ["..Q.", "Q...", "...Q", ".Q.."]]

counts = [len(solve_n_queens(n)) for n in range(1, 9)]
print("solution counts for n = 1..8:", counts)
assert counts == [1, 0, 0, 2, 10, 4, 40, 92]          # the classic sequence (OEIS A000170)

4-queens solutions:
    ['.Q..', '...Q', 'Q...', '..Q.']
    ['..Q.', 'Q...', '...Q', '.Q..']
solution counts for n = 1..8: [1, 0, 0, 2, 10, 4, 40, 92]


## 9. Measuring pruning -- does checking before you recurse actually save calls?

Same Combination Sum, two ways: **postcheck** (recurse into INCLUDE unconditionally, let the `remaining < 0`
case get caught as its own base case one call later) vs **precheck** (the version used above -- never make
the INCLUDE call at all if it would overshoot). Count the recursive calls.

In [10]:
def combination_sum_counted(candidates, target, mode):
    """Same algorithm as combination_sum, with a call counter and a switchable check."""
    out, path, calls = [], [], 0
    def backtrack(i, remaining):
        nonlocal calls
        calls += 1
        if remaining == 0:
            out.append(path[:]); return
        if i == len(candidates):
            return
        if mode == "postcheck":
            if remaining < 0:
                return
            path.append(candidates[i]); backtrack(i, remaining - candidates[i]); path.pop()
        else:                                             # "precheck"
            if remaining - candidates[i] >= 0:
                path.append(candidates[i]); backtrack(i, remaining - candidates[i]); path.pop()
        backtrack(i + 1, remaining)
    backtrack(0, target)
    return calls, out


cands, target = [2, 3, 5, 7], 20
post_calls, post_out = combination_sum_counted(cands, target, "postcheck")
pre_calls,  pre_out  = combination_sum_counted(cands, target, "precheck")
print(f"calls with postcheck (recurse, then bail): {post_calls}")
print(f"calls with precheck  (never make the doomed call): {pre_calls}")
print(f"answers found: {len(pre_out)} (identical in both modes)")
assert pre_calls < post_calls                             # the precheck genuinely removed work
assert sorted(map(sorted, post_out)) == sorted(map(sorted, pre_out))   # ...without changing the answer
assert sorted(map(sorted, pre_out)) == sorted(map(sorted, combination_sum(cands, target)))  # matches the real function

calls with postcheck (recurse, then bail): 491
calls with precheck  (never make the doomed call): 378
answers found: 18 (identical in both modes)


## ✅ Recap

- **One template:** choose -> explore -> un-choose. Pruning checks go *before* the choose line; the un-choose
  line is an exact mirror of the choose line (every piece of state, not just `path`).
- **Binary choice at a fixed element** (Subsets, Subsets II, Combinations, Combination Sum, Combination Sum
  II) -> **include/exclude**: try `backtrack` with the element in the path, then again without it.
- **Not a binary choice** -- order matters (Permutations), or the choice is "which of several options"
  (Word Search's 4 directions, Partitioning's cut length, Letter Combinations' letters, N-Queens' columns)
  -> stays a `used`-array or a direct loop over the options.
- **Reuse allowed** -> INCLUDE recurses with `i`; **each once** -> `i + 1`.
- **Duplicate input values** -> sort, then the EXCLUDE branch skips every remaining copy of the same value
  in one go (`while j < n and nums[j] == nums[i]: j += 1`), not just `i + 1`.
- **Check before you recurse** -> `if remaining - candidates[i] >= 0:` guards the INCLUDE call so the
  recursion never even enters an already-doomed branch (measured in section 9).
- **Grids** -> mark the cell itself, recurse 4 ways, restore. **Find-one** questions return `True` up the chain.
- **N-Queens** -> three sets (`cols`, `r - c`, `r + c`) make the safety check `O(1)`.
- **Always record a copy** (`path[:]`). Time is (number of arrangements) x (cost to copy one).

**Coverage:** Blind 75 -- [LC 39](https://leetcode.com/problems/combination-sum/), [LC 79](https://leetcode.com/problems/word-search/). NeetCode 150 -- those two plus [LC 78](https://leetcode.com/problems/subsets/),
[LC 90](https://leetcode.com/problems/subsets-ii/), [LC 40](https://leetcode.com/problems/combination-sum-ii/), [LC 46](https://leetcode.com/problems/permutations/), [LC 131](https://leetcode.com/problems/palindrome-partitioning/), [LC 17](https://leetcode.com/problems/letter-combinations-of-a-phone-number/),
[LC 51](https://leetcode.com/problems/n-queens/). Bonus: [LC 77](https://leetcode.com/problems/combinations/), [LC 47](https://leetcode.com/problems/permutations-ii/).

Prerequisites: [`04_Tree_Traversal`](../04_Tree_Traversal/README.md) (recursive DFS) and
[`07_Graph_Traversal`](../07_Graph_Traversal/README.md) (DFS on a grid).